# 03 — Summarizer

**Module notebook — definitions only.**

Map-reduce summarization over the full transcript, plus title generation.

Depends on: `get_llm()` (loaded in `00_llm_config.ipynb`).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Chunking config — only used as a FALLBACK now, for transcripts too long to
# fit in a single call even with the LLM's 262K token context window (see
# TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT in 00_llm_config.ipynb). For anything
# under that limit (in practice, tens of hours of speech), we summarize the
# whole transcript in one call instead — better coherence, fewer LLM calls,
# and no risk of losing cross-chunk context (e.g. a decision made early on
# being referenced again near the end).
SUMMARY_CHUNK_SIZE = 3000
SUMMARY_CHUNK_OVERLAP = 200


In [ ]:
def split_transcript(transcript: str) -> list:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=SUMMARY_CHUNK_SIZE,
        chunk_overlap=SUMMARY_CHUNK_OVERLAP,
    )
    return splitter.split_text(transcript)


In [ ]:
def _summarize_single_call(transcript: str, lang_instruction: str) -> str:
    """Summarize the whole transcript in one LLM call. Safe whenever the
    transcript is under TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT (see
    00_llm_config.ipynb) — which, for realistic meeting/video lengths, is
    almost always the case given this model\'s 262K token context window."""
    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are an expert meeting summarizer. Read the full meeting "
                "transcript below and write one professional summary in bullet "
                "points, covering the whole meeting from start to finish.\n\n"
                + lang_instruction,
            ),
            ("human", "{text}"),
        ]
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": transcript})


def _summarize_map_reduce(transcript: str, lang_instruction: str) -> str:
    """Fallback for transcripts that exceed the single-call limit: summarize
    in chunks, then combine those partial summaries into one final summary.
    Kept as a safety net, not the default path — map-reduce can lose some
    cross-chunk context (e.g. a decision revisited much later in the
    meeting), which a single full-context call doesn\'t suffer from."""
    llm = get_llm()

    map_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Summarize this portion of a meeting transcript concisely.\n\n"
                + lang_instruction,
            ),
            ("human", "{text}"),
        ]
    )
    map_chain = map_prompt | llm | StrOutputParser()

    chunks = split_transcript(transcript)
    chunk_summaries = [map_chain.invoke({"text": chunk}) for chunk in chunks]
    combined = "\n\n".join(chunk_summaries)

    combined_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are an expert meeting summarizer. Combine these partial summaries "
                "into one final professional meeting summary in bullet points.\n\n"
                + lang_instruction,
            ),
            ("human", "{text}"),
        ]
    )
    combined_chain = (
        RunnablePassthrough() | RunnableLambda(lambda x: {"text": x}) | combined_prompt | llm | StrOutputParser()
    )

    return combined_chain.invoke(combined)


def summarize(transcript: str, language: str = "english") -> str:
    lang_instruction = output_language_instruction(language)

    if len(transcript) <= TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT:
        return _summarize_single_call(transcript, lang_instruction)

    print(
        f"summarize: transcript ({len(transcript):,} chars) exceeds the "
        f"single-call limit ({TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT:,} chars) — "
        "falling back to chunked map-reduce summarization."
    )
    return _summarize_map_reduce(transcript, lang_instruction)


In [ ]:
def generate_title(transcript: str, language: str = "english") -> str:
    llm = get_llm()
    lang_instruction = output_language_instruction(language)

    title_chain = (
        RunnablePassthrough() | RunnableLambda(lambda x: {"text": x}) |
        ChatPromptTemplate.from_messages([
            (
                "system",
                "Based on the meeting transcript, generate a short professional meeting title "
                "(max 8 words). Only return the title, nothing else.\n\n"
                + lang_instruction,
            ),
            ("human", "{text}"),
        ])
        | llm
        | StrOutputParser()
    )

    return title_chain.invoke(transcript[:2000])
